# **AI Resume Screening System with LangChain & LangSmith**

## Objective
Build an AI-powered Resume Screening System that evaluates candidates based on job descriptions using LLM pipelines.

## Key Features
- Skill Extraction
- Resume Matching
- Scoring System (0–100)
- Explainable AI Output
- LangSmith Tracing for Debugging

In [2]:
!pip install -q langchain langchain-openai langchain-community langsmith transformers

## **Import Libraries**

In [3]:
import os
import json
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

## **Enable LangSmith**

In [4]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Resume-Screening-System"
os.environ["LANGCHAIN_API_KEY"] = "Your Key"

## **Job Description**

In [5]:
job_description = """
Role: Data Scientist

Skills: Python, Machine Learning, Deep Learning, SQL, Data Visualization
Experience: 2+ years
Tools: TensorFlow, Pandas, Scikit-learn
"""

## **Input Resumes**

In [6]:
strong_resume = "Python, Machine Learning, Deep Learning, SQL, TensorFlow | 3 years experience"
average_resume = "Python, SQL, basic ML | 1 year experience"
weak_resume = "HTML, CSS, Excel | no relevant experience"

# **Initialize LLM**

In [22]:
!pip uninstall -y transformers
!pip install transformers==4.36.2

Found existing installation: transformers 4.36.2
Uninstalling transformers-4.36.2:
  Successfully uninstalled transformers-4.36.2
  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
Using cached transformers-4.36.2-py3-none-any.whl (8.2 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.


In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

hf_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.5,
    top_p=0.9,
    device=-1
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### **MASTER PROMPT**

In [21]:
main_prompt = PromptTemplate(
    input_variables=["resume", "job"],
    template="""
Instruction: Analyze the resume and generate full evaluation.

Example Output:
Skills: Python, SQL
Tools: Pandas
Experience: 2 years

Matching Skills: Python
Missing Skills: Deep Learning

Score: 70
Reason: Partial match

Strengths:
- Good basics
- Some experience

Weaknesses:
- Missing advanced skills
- Low experience

Final Decision: Selected

Now analyze:

Resume: {resume}
Job: {job}

Output:
"""
)

### **Core Function**

In [22]:
def calculate_score(resume, job_description):
    required_skills = ["python", "machine learning", "deep learning", "sql", "data visualization"]

    resume_text = resume.lower()

    matched = [skill for skill in required_skills if skill in resume_text]
    missing = [skill for skill in required_skills if skill not in resume_text]

    score = int((len(matched) / len(required_skills)) * 100)

    return matched, missing, score

In [37]:
def evaluate_resume(resume):
    matched, missing, score = calculate_score(resume, job_description)

    # Strengths (rule-based)
    strengths = []
    if matched:
        strengths.append(f"Has relevant skills: {', '.join(matched)}")
    if score >= 70:
        strengths.append("Strong alignment with job requirements")
    elif score >= 40:
        strengths.append("Basic understanding of required skills")
    else:
        strengths.append("Some general technical exposure")


    while len(strengths) < 2:
        if len(strengths) == 0:
            strengths.append("No primary strengths identified")
        else: # len(strengths) == 1
            strengths.append("Further potential identified")

    # Weaknesses (rule-based)
    weaknesses = []
    if missing:
        weaknesses.append(f"Missing important skills: {', '.join(missing)}")
    if score < 50:
        weaknesses.append("Insufficient experience for the role")
    else:
        weaknesses.append("Needs improvement in some areas")


    while len(weaknesses) < 2:
        if len(weaknesses) == 0:
            weaknesses.append("No significant weaknesses apparent")
        else:
            weaknesses.append("Minor areas for development")

    # Final Decision
    if score > 70:
        decision = "Selected"
    elif score >= 40:
        decision = "Consider"
    else:
        decision = "Rejected"

    return f"""
Matching Skills: {matched}
Missing Skills: {missing}

Score: {score}

Strengths:
- {strengths[0]}
- {strengths[1]}

Weaknesses:
- {weaknesses[0]}
- {weaknesses[1]}

Final Decision: {decision}
"""

### **Output**

In [38]:
def display_result(result, title):
    print(f"\n===== {title} =====")
    print(result)

### **Run All Candidates**

In [39]:
display_result(evaluate_resume(strong_resume), "STRONG CANDIDATE")
display_result(evaluate_resume(average_resume), "AVERAGE CANDIDATE")
display_result(evaluate_resume(weak_resume), "WEAK CANDIDATE")


===== STRONG CANDIDATE =====

Matching Skills: ['python', 'machine learning', 'deep learning', 'sql']
Missing Skills: ['data visualization']

Score: 80

Strengths:
- Has relevant skills: python, machine learning, deep learning, sql
- Strong alignment with job requirements

Weaknesses:
- Missing important skills: data visualization
- Needs improvement in some areas

Final Decision: Selected


===== AVERAGE CANDIDATE =====

Matching Skills: ['python', 'sql']
Missing Skills: ['machine learning', 'deep learning', 'data visualization']

Score: 40

Strengths:
- Has relevant skills: python, sql
- Basic understanding of required skills

Weaknesses:
- Missing important skills: machine learning, deep learning, data visualization
- Insufficient experience for the role

Final Decision: Consider


===== WEAK CANDIDATE =====

Matching Skills: []
Missing Skills: ['python', 'machine learning', 'deep learning', 'sql', 'data visualization']

Score: 0

Strengths:
- Some general technical exposure
- Furt

## **Debug Case**

In [40]:
debug_resume = "I am expert in everything"

display_result(evaluate_resume(debug_resume), "DEBUG CASE")


===== DEBUG CASE =====

Matching Skills: []
Missing Skills: ['python', 'machine learning', 'deep learning', 'sql', 'data visualization']

Score: 0

Strengths:
- Some general technical exposure
- Further potential identified

Weaknesses:
- Missing important skills: python, machine learning, deep learning, sql, data visualization
- Insufficient experience for the role

Final Decision: Rejected



## Conclusion

This project demonstrates an AI-powered resume screening system using LangChain.

### Key Achievements
- Modular LLM pipeline
- Skill extraction & matching
- Scoring with explainability
- LangSmith tracing enabled

### Limitations
- Output depends on prompt quality
- LLM inconsistency possible

### Future Improvements
- JSON structured output
- Weighted scoring system
- Real-world dataset integration
- UI-based recruiter dashboard